# 🩺 Huấn luyện Mamba-UNet (MambaVision Backbone) trên Kaggle Environment

Quy trình: Clone/đồng bộ mã nguồn GitHub -> Dò tìm dataset trong /kaggle/input (kể cả nested path cache_1mm/cache_1mm) -> Dựng symlink view-dir ghi được trong /kaggle/working -> Huấn luyện -> Đánh giá 3D.

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi

In [ ]:
# 2. Tải và đồng bộ mã nguồn từ GitHub (Yêu cầu bật Internet: ON trong Settings bên phải)
import os
import sys

GITHUB_REPO = 'https://github.com/NguyenMinhTri24072005/CT208E_LIDC_DuneAI_MambaVision.git'
BRANCH = 'main'  # Nhánh mặc định trên GitHub (đổi nếu bạn dùng nhánh khác)
REPO_DIR = '/kaggle/working/CT208E_LIDC_DuneAI_MambaVision'

if os.path.exists('mamba_unet') and os.path.exists('common'):
    print(f'✅ Đang chạy trực tiếp trong thư mục dự án: {os.getcwd()}')
else:
    if not os.path.exists(REPO_DIR):
        print(f'📥 Đang clone repository từ GitHub (nhánh {BRANCH})...')
        !git clone -b {BRANCH} {GITHUB_REPO} {REPO_DIR}
    else:
        print(f'🔄 Đang đồng bộ mã nguồn mới nhất từ GitHub...')
        %cd {REPO_DIR}
        !git fetch origin
        !git checkout {BRANCH}
        !git pull origin {BRANCH}
    %cd {REPO_DIR}

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'🚀 Thư mục làm việc hiện tại: {os.getcwd()}')

In [ ]:
# 3. Cài đặt các thư viện cần thiết
!pip install -q SimpleITK pyyaml opencv-python-headless

In [ ]:
# 4. Chuẩn bị dữ liệu Kaggle Input qua Symlink View-Dir
# Hỗ trợ đường dẫn trực tiếp hoặc tự động dò tìm dưới /kaggle/input
from common.data_pipeline.runtime_data import run_kaggle

# Đường dẫn dataset mẫu hoặc thư mục cha chứa dataset
KAGGLE_HINT = '/kaggle/input/datasets/minhtri2475/lidc-idri-1mm-cache-ct208e/cache_1mm/cache_1mm'
VIEW_DIR = '/kaggle/working/data/cache_1mm'

cache_dir = run_kaggle(
    hint=KAGGLE_HINT,
    view_dir=VIEW_DIR,
    n_patients=0,  # 0 = toàn bộ bệnh nhân có trong dataset
)
print(f'✅ Dữ liệu sẵn sàng tại: {cache_dir}')

In [ ]:
# 5. Bắt đầu quá trình huấn luyện
!python -m mamba_unet.train_main train \
    --cache-dir /kaggle/working/data/cache_1mm \
    --backbone mamba_vision_T \
    --pretrained \
    --freeze-backbone-epochs 5 \
    --backbone-lr-mult 0.1 \
    --epochs 50 \
    --batch-size 8 \
    --lr 1e-4 \
    --tag kaggle_run

In [ ]:
# 6. Đánh giá 3D cấp độ bệnh nhân trên tập Validation
!python -m mamba_unet.evaluate eval \
    --cache-dir /kaggle/working/data/cache_1mm \
    --checkpoint checkpoints/best_model.pth \
    --split val